In [3]:
import networkx as nx

In [4]:
def move(grid, pos, direction):
    # Diagonal case
    if 0 not in direction:
        r, c = pos
        new_pos = (r + direction[0], c + direction[1])
        while "w" not in [grid[new_pos[0]][new_pos[1]], grid[new_pos[0]-direction[0]][new_pos[1]], grid[new_pos[0]][new_pos[1]-direction[1]]]:
            r, c = new_pos
            new_pos = (r+direction[0], c+direction[1])
        return (new_pos[0] - direction[0], new_pos[1] - direction[1])
        
    # Cardinal case
    r, c = pos
    new_pos = (r + direction[0], c+direction[1])
    while "w" not in [grid[new_pos[0]][new_pos[1]]]:
        r, c = new_pos
        new_pos = (r + direction[0], c + direction[1])
    return (new_pos[0] - direction[0], new_pos[1] - direction[1])

Directions:
- (1,0) is south
- (0,1) is east
- (-1,0) is north
- (0,-1) is west

In [5]:
import matplotlib.pyplot as plt

def show_graph(G):
    subax1 = plt.subplot(121)
    nx.draw(G, with_labels=True, font_weight='bold')

In [7]:
# pos = (r, c)
def build_bear_graph(grid, starting_pos):
    DG = nx.DiGraph()
    current_node = starting_pos
    while True:
        # print("Looping with node " + str(current_node))
        # Add all edges from current node
        for x in [-1,0,1]:
            for y in [-1,0,1]:
                if [x,y] != [0,0]:
                    new_node = move(grid, current_node, (x,y))
                    if new_node != current_node:
                        DG.add_edge(current_node, move(grid, current_node, (x, y)))
        
        # Find next node
        node_found = False
        # print(DG.nodes)
        for node in DG.nodes:
            # print("Edges of node " + str(node) + " are " + str(DG.edges(node)))
            if len(DG.edges(node)) == 0:
                current_node = node
                node_found = True
                break
        if not node_found:
            break
            
    return DG

TSP:
- Need a shortest path algorithm first
- Find shortest path between every pair of nodes
- Turn this into a weighted graph
- Find shortest path that covers every vertex


Shortest path algorithm:
- Pick a node
- Extend path in every direction from that node
- When a new shortest path is found, store it and its length.
- End when picked node is target node


Recursion?
- Shortest path from a to b is shortest (path from neighbour of a to b + 1)
- Need to keep track of which nodes to ignore, to avoid infinite loops

In [10]:
def build_paths(paths, graph):
    out = []
    for path in paths:
        neighbors = [edge[1] for edge in graph.edges(path[-1])]
        for neighbor in neighbors:
            if neighbor not in path:
                out.append(path + [neighbor])
    return out


def shortest_path(a, b, graph):
    paths = [[a]]
    while True:
        for path in paths:
            if path[-1] == b:
                return path
        paths = build_paths(paths, graph)


In [ ]:
# Paintings graph is built off the back of the bear graph
# Node set is paintings, with starting position
# A near-complete directed graph (doesn't need paths going to the starting node)
# Each edge has a 'path' attribute which corresponds to the path in the original graph
# Each edge has a 'length' attribute which correspond to the length of its path

def paintings_graph(grid, paintings, start):
    bear_graph = build_bear_graph(grid, start)
    # Set up graph and nodes
    paintings_graph = nx.DiGraph()
    paintings_graph.add_node(start)
    paintings_graph.add_nodes_from(paintings)

    # Set up edges
    for a in paintings_graph.nodes:
        for b in paintings:
            if a != b:
                new_edge_path = shortest_path(a, b, bear_graph)
                new_edge_length = len(new_edge_path) - 1         # -1 because e.g. a path of length 3 requires 2 moves.
                paintings_graph.add_edge(a, b)
                paintings_graph[a][b]['length'] = new_edge_length
                paintings_graph[a][b]['path'] = new_edge_path

    return paintings_graph

In [79]:
from copy import copy

def find_solution(grid, paintings, start):
    PG = paintings_graph(grid, paintings, start)
    paths = [[start]]
    # Find all Hamiltonian paths
    i = 0
    while True:
        paths = build_paths(paths, PG)
        if len(paths[0]) == len(paintings) + 1 or i == 6:
            break
        i += 1

    # Find lengths of Hamiltonian paths
    length = {}
    for path in paths:
        length[tuple(path)] = 0
        for i in range(len(path) - 1):
            length[tuple(path)] += PG[path[i]][path[i+1]]['length']

    # Find shortest Hamiltonian path
    min_length = min(length.values())
    for key in length.keys():
        if length[key] == min_length:
            min_path = list(key)

    # Open up the path
    # print(min_path)
    # print(PG.nodes)
    bear_path = copy(PG[min_path[0]][min_path[1]]['path'])
    for i in range(1, len(min_path) - 1):
        new = copy(PG[min_path[i]][min_path[i+1]]['path'])
        new.pop(0)
        bear_path += new

    return bear_path

In [70]:
def paintings_coords(grid):
    out = []
    for r, row in enumerate(grid):
        for c, tile in enumerate(row):
            if tile == "p":
                out += [(r,c)]
    if out == []:
        raise ValueError('No paintings found in grid.')
    return out

def start_coords(grid):
    for r, row in enumerate(grid):
        for c, tile in enumerate(row):
            if tile == "s":
                return (r,c)
    raise ValueError('No starting position found in grid.')

# Examples

In [72]:
grid = [["w", "w", "w", "w", "w"],
        ["w", "s", ".", ".", "w"],
        ["w", "w", ".", ".", "w"],
        ["w", ".", ".", "p", "w"],
        ["w", ".", "p", "w", "w"],
        ["w", ".", "p", "w", "w"],
        ["w", "w", "w", "w", "w"]]
print(find_solution(grid, paintings_coords(grid), start_coords(grid)))

[(1, 1), (1, 3), (3, 3)]
[(1, 1), (1, 3), (3, 3), (3, 1), (4, 2), (5, 2)]
[(1, 1), (1, 3), (3, 3), (3, 1), (4, 2), (5, 2)]


In [78]:
grid = [
    ["w", "w", "w", "w", "w", "w"],
    ["w", "w", "s", ".", "w", "w"],
    ["w", "w", ".", ".", "w", "w"],
    ["w", ".", ".", ".", "p", "w"],
    ["w", "p", ".", ".", "p", "w"],
    ["w", "p", ".", ".", ".", "w"],
    ["w", "w", ".", ".", "w", "w"],
    ["w", "w", "w", "w", "w", "w"]]
print(find_solution(grid, paintings_coords(grid), start_coords(grid)))

[(1, 2), (6, 2), (4, 4)]
[(1, 2), (6, 2), (4, 4), (4, 1), (5, 1), (3, 1), (3, 4)]
[(1, 2), (6, 2), (4, 4), (4, 1), (5, 1), (3, 1), (3, 4)]


In [75]:
grid = [
    ["w", "w", "w", "w", "w", "w", "w", "w"],
    ["w", "w", "w", "s", ".", "w", "w", "w"],
    ["w", "w", "w", ".", ".", "w", "w", "w"],
    ["w", ".", ".", ".", ".", ".", ".", "w"],
    ["w", ".", "p", ".", ".", ".", ".", "w"],
    ["w", "w", "w", ".", "p", "w", "w", "w"],
    ["w", ".", ".", ".", ".", ".", ".", "w"],
    ["w", ".", "p", ".", ".", "p", ".", "w"],
    ["w", "w", "w", ".", ".", "w", "w", "w"],
    ["w", "w", "w", "w", "w", "w", "w", "w"]]
print(find_solution(grid, paintings_coords(grid), start_coords(grid)))

[(1, 3), (2, 4), (4, 2)]
[(1, 3), (2, 4), (4, 2), (3, 2), (5, 4), (7, 2), (5, 4), (5, 3), (7, 5)]
[(1, 3), (2, 4), (4, 2), (3, 2), (5, 4), (7, 2), (5, 4), (5, 3), (7, 5)]


In [77]:
grid = [["w", "w", "w", "w", "w", "w", "w"], 
        ["w", "w", ".", ".", ".", ".", "w"], 
        ["w", "w", "p", ".", ".", ".", "w"], 
        ["w", "s", ".", "w", "w", ".", "w"], 
        ["w", ".", ".", "p", ".", "p", "w"], 
        ["w", ".", ".", ".", "w", "w", "w"], 
        ["w", ".", ".", "w", "w", "w", "w"], 
        ["w", "p", ".", "w", "w", "w", "w"], 
        ["w", "w", "w", "w", "w", "w", "w"]]
print(find_solution(grid, paintings_coords(grid), start_coords(grid)))

[(3, 1), (7, 1)]
[(3, 1), (7, 1), (3, 1), (5, 3), (4, 3), (4, 5), (1, 5), (2, 4), (2, 2)]
[(3, 1), (7, 1), (3, 1), (5, 3), (4, 3), (4, 5), (1, 5), (2, 4), (2, 2)]
